# Entrenamiento Mechatronics AI en la nube (Colab / GPU)

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)*.

**Preparación en tu PC:**
1. `python scripts/split_dataset.py` (genera `dataset/train`, `val`, `test`).
2. Comprime la carpeta `dataset/` (con `data.yaml`, `train/`, `val/`, `test/`) como `dataset.zip`.
3. Súbela a Google Drive en `MyDrive/mechatronics-ai/dataset.zip`.

El entrenamiento usa `scripts/train.py` del repo, con la misma augmentation que en local (color, sombras, geometría y cámara barata).

## 1. Google Drive y código

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/mechatronics-ai'
REPO = 'https://github.com/issxccrrll/mechatronics-ai.git'
RAMA = 'main'  # o la rama con los últimos cambios, ej. 'camara-mejoras'

!rm -rf /content/mechatronics-ai && git clone -q -b {RAMA} {REPO} /content/mechatronics-ai
%cd /content/mechatronics-ai
!pip install -q ultralytics albumentations
import ultralytics; ultralytics.checks()

## 2. Copiar el dataset al disco local de Colab

Leer miles de fotos directo de Drive es muy lento; copiar el zip y descomprimirlo es mucho más rápido.

In [ ]:
!cp {BASE}/dataset.zip /content/ && unzip -q -o /content/dataset.zip -d /content/
!ls /content/dataset && for s in train val test; do echo $s: $(ls /content/dataset/$s/images | wc -l) imágenes; done

## 3. Entrenar

- `--batch -1`: tamaño de lote automático según la memoria de la GPU.
- `--cache ram`: carga las fotos en RAM para épocas más rápidas.
- Los resultados se guardan en Drive (`runs/`): si Colab se desconecta no se pierden.
- Para comparar sin la augmentation extra, agrega `--sin-aug-extra` y cambia `--name`.

In [ ]:
!python scripts/train.py --data /content/dataset/data.yaml --device 0 --epochs 80 --batch -1 --cache ram --project {BASE}/runs --name mechatronics

## 4. Evaluar en test (incluye ejemplares no vistos si se reservaron con `--ejemplares-test`)

In [ ]:
!python scripts/eval.py --model {BASE}/runs/mechatronics/weights/best.pt --data /content/dataset/data.yaml
!cp metricas.md metricas.csv {BASE}/ && mkdir -p {BASE}/docs && cp docs/confusion_matrix.png {BASE}/docs/ 2>/dev/null; echo copiado a Drive

## 5. Descargar el modelo

Guarda `best.pt` en `models/` de tu PC y exporta allí a OpenVINO (la exportación conviene hacerla en la CPU donde correrá la demo):

`python scripts/export.py --model models/best.pt`

In [ ]:
from google.colab import files
files.download(f'{BASE}/runs/mechatronics/weights/best.pt')